In [1]:
import numpy as np
import pandas as pd
import os
from sklearn.metrics import mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping
import joblib

# === CONFIG ===
look_back = 24
data_dir = r"C:\Users\Muhammad Luqman\Desktop\open-ended-lab-2nd methed"
target_col = 0  # Index of the target column

# ===  Data Load  ===
train = pd.read_csv(os.path.join(data_dir, "train_data.csv"))
val = pd.read_csv(os.path.join(data_dir, "val_data.csv"))
test = pd.read_csv(os.path.join(data_dir, "test_data.csv"))

# ===  Sequences createing  ===
def create_sequences(data, look_back, target_col):
    X, y = [], []
    for i in range(len(data) - look_back):
        X.append(data.iloc[i:i+look_back].values)
        y.append(data.iloc[i+look_back, target_col])
    return np.array(X), np.array(y)

X_train, y_train = create_sequences(train, look_back, target_col)
X_val, y_val = create_sequences(val, look_back, target_col)
X_test, y_test = create_sequences(test, look_back, target_col)

# === Build LSTM Model  and compile ===
model = Sequential([
    LSTM(64, activation='tanh', input_shape=(look_back, X_train.shape[2])),
    Dense(32, activation='relu'),
    Dense(1)
])
model.compile(optimizer='adam', loss='mse')

# === Train ===
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

# === Evaluate ===
val_loss = model.evaluate(X_val, y_val)
test_loss = model.evaluate(X_test, y_test)
print(f"Validation RMSE: {np.sqrt(val_loss):.4f}")
print(f"Test RMSE: {np.sqrt(test_loss):.4f}")

# === Save Model ===
model.save(os.path.join(data_dir, "lstm_model.h5"))
print("✅ LSTM model saved.")


Epoch 1/10
191/191 [==============================] - 7s 19ms/step - loss: 0.0037 - val_loss: 0.0026
Epoch 2/10
191/191 [==============================] - 3s 15ms/step - loss: 6.0639e-04 - val_loss: 0.0017
Epoch 3/10
191/191 [==============================] - 3s 16ms/step - loss: 3.2974e-04 - val_loss: 0.0014
Epoch 4/10
191/191 [==============================] - 3s 16ms/step - loss: 2.2033e-04 - val_loss: 0.0013
Epoch 5/10
191/191 [==============================] - 3s 16ms/step - loss: 1.6174e-04 - val_loss: 0.0011
Epoch 6/10
191/191 [==============================] - 3s 15ms/step - loss: 1.2694e-04 - val_loss: 0.0011
Epoch 7/10
191/191 [==============================] - 3s 15ms/step - loss: 1.0663e-04 - val_loss: 0.0010
Epoch 8/10
191/191 [==============================] - 3s 16ms/step - loss: 1.0003e-04 - val_loss: 8.1426e-04
Epoch 9/10
191/191 [==============================] - 3s 15ms/step - loss: 7.3017e-05 - val_loss: 7.5059e-04
Epoch 10/10
27/27 [==============================] 